Settings

In [ ]:
DATASET = "oracle_cards"
MODEL_PATH = "models"

Imports

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms
from torchvision import models
import torch.nn as nn


Read data manifest

In [ ]:
df = pd.read_csv(f"data/{DATASET}_manifest.csv")
print(f"Total cards: {len(df)}")
print(df.head())
print(df["types"].value_counts().head(20))

Build dataset

In [ ]:
class CreatureDataset(Dataset):
    def __init__(self, df, all_types, transform=None):
        self.df = df.reset_index(drop=True)
        self.all_types = all_types
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        label = torch.zeros(len(self.all_types))
        types = row["types"].split("|") if isinstance(row["types"], str) else []
        for t in types:
            if t in self.all_types:
                label[self.all_types.index(t)] = 1.0
        
        return image, label

In [ ]:
all_types = sorted(set(
    t for types_str in df["types"] if isinstance(types_str, str) for t in types_str.strip().split("|")
))
print(all_types)
print(f"amount of creature types:", len(all_types))

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = CreatureDataset(df, all_types, transform=transform)
print(f"Dataset size: {len(dataset)}")

image, label = dataset[0]
print(f"Image tensor shape: {image.shape}")
print(f"Label vector shape: {label.shape}")
print(f"Active labels: {[all_types[i] for i, v in enumerate(label) if v == 1.0]}")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

Model

In [ ]:
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.backends.cuda.matmul.allow_tf32 = True
print(torch.cuda.get_arch_list())
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(all_types))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)

print(torch.__version__)
print(torch.version.cuda)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# we freeze all except the last layers
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

Training

In [ ]:
import os
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

num_epochs = 10
for epoch in range(num_epochs):
    # train
    model.train()
    train_loss = 0.0
    scaler = torch.amp.GradScaler('cuda')
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} train"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} val"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} — train loss: {avg_train_loss:.4f} — val loss: {avg_val_loss:.4f}")
    
    path = Path(f"{MODEL_PATH}/model_epoch_{epoch+1}.pth")
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), "models/model_epoch4.pth")